# 08X · Can discovery without marker genes separate lineage from state?

**Optional companion to 08W** · Adagrasib mouse Visium HD · 1.0.0  
**Run:** place this notebook beside the working 08V notebook. Select a fresh GPU kernel in the UV environment with **scvi-tools, Pyro, PyTorch and CUDA**, then **Run All**. You need the completed 08V files on EC2. 08W does not have to have run.

Each informative T cell has fractional membership in **12 expression programs** learned by the scvi-tools RNA AmortizedLDA model. A shared state and a lineage could both contribute to the same cell. The model does not identify which program, if any, represents CD4 or CD8 in advance. Eight unlabeled clusters of these mixtures provide one exploratory display; neither 12 nor eight is the expected number of cell identities.

### Experiment
1. Reserve one entire sample per condition before fitting. Draw up to 4,000 cells per remaining sample, without looking at expression or markers. In the 12-sample study this is up to **32,000 training cells** and **four evaluation samples**.
2. Use original molecule counts. Select genes detected in ≥0.1% of discovery-fit cells (minimum ten). Reserve **Cd4, Cd8a, Cd8b1, Zbtb7b, Cd40lg, Foxp3, Il2ra, Ctla4, Ikzf2, Tnfrsf18**. No ratio weights, HVG ranking, spatial smoothing or regression. Stress and exhaustion genes are eligible to define topics.
3. Fit one model on one GPU for 35 epochs; project all informative cells. Rows lacking counts in the fitting features stay in the output with missing memberships. Inputs are sparse and inference is batched. There is no explicit sample integration; look for sample-specific and depth-associated topics.
4. Measure whether frozen topics add lineage information above depth. A separate evaluation classifier learns only from observed CD4-only and CD8-only cells in discovery samples; it is evaluated by sample and depth bin in the withheld samples. It does not change topics, assign lineage labels to other cells, or influence the eight clusters.

**Runtime:** independent of 08W; full-data EC2 runtime is not yet measured. Compatible finished models/projections are cached. Only the compact summary ZIP needs to be returned.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path.cwd()
SOURCE_08V_RUN = None   # normally auto-found; optional completed EC2 08V folder containing comparison_complete.json

### Restore the standalone code

The collapsed cell restores and verifies the packaged source without installing software.

In [ ]:
import base64,hashlib,importlib,json,os,shutil,sys,traceback,zipfile,zlib
from IPython.display import display,FileLink,Image
import pandas as pd
_raw=zlib.decompress(base64.b64decode('...')); _id='6dfab1b9705a9a231645ef666af03023af89f6bbc0d81309a010791105ddbb1a'
if hashlib.sha256(_raw).hexdigest()!=_id: raise ValueError("Incomplete 08X helper payload; please download the notebook again.")
_help=(Path.cwd()/".08X_helpers"/_id[:16]).resolve();_help.mkdir(parents=True,exist_ok=True)
for _f,_s in json.loads(_raw).items():
    _p=_help/_f
    if not _p.exists() or _p.read_text(encoding="utf8")!=_s: _p.write_text(_s,encoding="utf8")
    sys.modules.pop(Path(_f).stem,None)
sys.path.insert(0,str(_help));importlib.invalidate_caches()
def make_failure(exc):
    dst=Path.cwd()/"08X_failure_diagnostics.zip"
    with zipfile.ZipFile(dst,"w",zipfile.ZIP_DEFLATED) as z:
        z.writestr("error.txt","".join(traceback.format_exception(type(exc),exc,exc.__traceback__)))
        z.writestr("context.json",json.dumps({"python":sys.executable,"project":str(PROJECT_DIR),"source_08v_run":str(SOURCE_08V_RUN),"helper_sha256":_id},indent=2))
    display(FileLink(str(dst.name),result_html_prefix="Return diagnostics: "))
print("08X 1.0.0 ·",sys.executable)

## Run the RNA topic model and review

All outputs live separately from the original T-cell analysis. The full H5AD contains each original cell's mixture and unsupervised cluster; the report uses measured marker counts.

In [ ]:
try:
    import tcell_topic_pilot as tp
    result=tp.run(PROJECT_DIR,SOURCE_08V_RUN)
    folder=Path(result["run_root"])
    local=Path.cwd()/"08X_results"/folder.name;local.mkdir(parents=True,exist_ok=True)
    review=Path(result["review_dir"]); localzip=local/"08X_return_to_chat.zip"
    shutil.copy2(result["zip_path"],localzip)
    display(pd.read_csv(review/"sample_design.csv"))
    display(pd.read_csv(review/"primary_lineage_test.csv"))
    display(Image(filename=str(review/"topic_lineage_review.png"),width=1100))
    display(FileLink(os.path.relpath(localzip,Path.cwd()),result_html_prefix="Return this result ZIP: "))
    print("Full per-cell topic measurements:",folder/"all_cells_topics.h5ad")
except (Exception,KeyboardInterrupt) as e:
    make_failure(e)
    raise

### What to look for

Inspect **absolute AUROC and the gain over depth** in each held-out sample. An increase near 0.5 is weak evidence. Examine anchor counts in each depth stratum, the original marker fractions by cluster, and the top genes and depth association for each topic. Cells without coreceptor detection are not labeled by this test.

The number of topics and clusters is fixed for this pilot. A favorable result warrants additional seeds, sample splits, and validation before scaling to a whole tumor atlas. A weak result is useful too: an unsupervised model can recover strong cell states yet miss a sparse lineage axis. This experiment is conditional on previously identified T cells.

[Official AmortizedLDA RNA tutorial](https://docs.scvi-tools.org/en/stable/tutorials/notebooks/scrna/amortized_lda.html) · [API documentation](https://docs.scvi-tools.org/en/stable/api/reference/scvi.model.AmortizedLDA.html) · [Grade-of-membership models for sequencing counts](https://doi.org/10.1186/s13059-023-03067-9)